In [ ]:
import json
import os
from datetime import date
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
from time import sleep

load_dotenv()

LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://127.0.0.1:8321")
MODEL_ID = "vllm/openai/gpt-oss-20b"
MODEL_LABEL = "GPT-OSS-20b"
VLLM_VERSION = "0.11.2+rhai5"
RUN_DATE = os.getenv("RUN_DATE", date.today().isoformat())

MAX_TOOL_CALLS = 20
MAX_INFER_ITERS = 20

# Use llama-stack client (supports max_infer_iters parameter, unlike OpenAI client)
client = LlamaStackClient(base_url=LLAMA_STACK_URL)

# Note: Using llama-stack client to pass max_infer_iters directly (not supported by OpenAI client).
# max_infer_iters controls the total number of iterations (tool calls + final message).
# max_tool_calls limits the number of tool calls within those iterations.

PROMPTS = [
    "Use the file search tool to find information in the uploaded documents. What information is available?",
    "Search the uploaded documents and summarize the key points.",
    "What are the main topics covered in the uploaded documents? Use file search to find this information."
]

# Vector store setup - automatically create if needed
VECTOR_STORE_IDS = os.getenv("VECTOR_STORE_IDS", "").split(",") if os.getenv("VECTOR_STORE_IDS") else []
VECTOR_STORE_NAME = "file-search-reference-docs"
REFERENCE_FILE = "reference.txt"

# Create vector store and upload file if not already set
if not VECTOR_STORE_IDS or not VECTOR_STORE_IDS[0]:
    print("Setting up vector store...")
    
    # Check if reference file exists
    if not os.path.exists(REFERENCE_FILE):
        raise FileNotFoundError(f"Reference file '{REFERENCE_FILE}' not found. Please create it in the same directory as this notebook.")
    
    # Upload the file
    print(f"Uploading {REFERENCE_FILE}...")
    with open(REFERENCE_FILE, "rb") as f:
        file = client.files.create(file=f, purpose="assistants")
    print(f"✅ Uploaded file: {file.id}")
    
    # Create vector store
    print("Creating vector store...")
    vector_store = client.vector_stores.create(name=VECTOR_STORE_NAME)
    print(f"✅ Created vector store: {vector_store.id}")
    
    # Add file to vector store
    print("Adding file to vector store...")
    vector_store_file = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file.id
    )
    print(f"✅ Added file to vector store")
    
    # Wait for file processing
    print("Waiting for file processing...")
    max_wait = 60  # seconds
    waited = 0
    while waited < max_wait:
        sleep(2)
        waited += 2
        file_status = client.vector_stores.files.retrieve(
            vector_store_id=vector_store.id,
            file_id=file.id
        )
        if file_status.status == "completed":
            print(f"✅ File processed successfully after {waited}s")
            break
        elif file_status.status == "failed":
            print(f"❌ File processing failed after {waited}s")
            break
        if waited % 10 == 0:
            print(f"  Still processing... ({waited}s)")
    
    if file_status.status != "completed":
        print(f"⚠️  Warning: File status is {file_status.status} after {waited}s")
    
    VECTOR_STORE_IDS = [vector_store.id]
    print(f"\n✅ Vector store setup complete. ID: {VECTOR_STORE_IDS[0]}")
    print(f"   You can set VECTOR_STORE_IDS={VECTOR_STORE_IDS[0]} as an environment variable to reuse it.")
else:
    # Verify vector store exists
    try:
        vs = client.vector_stores.retrieve(vector_store_id=VECTOR_STORE_IDS[0])
        print(f"✅ Using existing vector store: {VECTOR_STORE_IDS[0]}")
        print(f"   Status: {vs.status}, Files: {vs.file_counts.total}")
    except Exception as e:
        print(f"❌ Error accessing vector store {VECTOR_STORE_IDS[0]}: {e}")
        print("   Please check the VECTOR_STORE_IDS or let the script create a new one.")
        raise

def run_prompt(prompt):
    print(f"\nUser: {prompt}")
    response = client.responses.create(
        model=MODEL_ID,
        tools=[{"type": "file_search", "vector_store_ids": VECTOR_STORE_IDS}],
        input=prompt,
        max_tool_calls=MAX_TOOL_CALLS,
        max_infer_iters=MAX_INFER_ITERS
    )

    print(f"Response ID: {response.id}")
    print(f"Response status: {response.status}")
    
    file_search_triggered = False
    tool_call_count = 0
    message_count = 0

    for item in response.output:
        item_type = getattr(item, "type", "unknown")
        item_status = getattr(item, "status", "unknown")
        print(f"  - Type: {item_type}, Status: {item_status}")
        if "file_search" in item_type:
            file_search_triggered = True
            tool_call_count += 1
        elif item_type == "message":
            message_count += 1

    print("\nDiagnostics:")
    print(f"  File search triggered: {file_search_triggered}")
    print(f"  Tool calls: {tool_call_count}")
    print(f"  Messages: {message_count}")
    
    print(f"  Total iterations: {tool_call_count + message_count}")
    
    # Check for max_infer_iters issue
    if response.status == "incomplete" and tool_call_count >= MAX_TOOL_CALLS and message_count == 0:
        print(f"  ⚠️  WARNING: Likely hit max_tool_calls limit ({MAX_TOOL_CALLS})")
        print(f"     Model used {tool_call_count} tool calls, leaving no iterations for final message")
    elif response.status == "incomplete" and (tool_call_count + message_count) >= MAX_INFER_ITERS:
        print(f"  ⚠️  WARNING: Likely hit max_infer_iters limit ({MAX_INFER_ITERS})")
        print(f"     Total iterations: {tool_call_count + message_count}")
    
    print("\n" + "=" * 60)
    print(f"Response from {MODEL_LABEL} with vLLM {VLLM_VERSION} ({RUN_DATE})")
    print("=" * 60)
    if response.output_text:
        print(response.output_text)
    else:
        print("(empty response)")
    return response


# Store all responses for JSON metadata
all_responses = []

try:
    for prompt in PROMPTS:
        response = run_prompt(prompt)
        all_responses.append(response)
        sleep(10)  # Brief pause between prompts
finally:
    # llama-stack client doesn't need explicit close, but keeping for consistency
    pass

# Generate summary and JSON, then update markdown cells
import json as json_lib

summary_md = "# Run Output Summary\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    file_search_triggered = any("file_search" in str(item.type) for item in response.output)
    tool_call_count = sum(1 for item in response.output if "file_search" in str(item.type))
    message_count = sum(1 for item in response.output if item.type == "message")
    
    summary_md += f"## Prompt {i}\n\n"
    summary_md += f"**Status:** {response.status} | **File search:** {file_search_triggered} | **Tool calls:** {tool_call_count} | **Messages:** {message_count}\n\n"
    output_text = response.output_text or "(empty response)"
    summary_md += f"**Response:** {output_text[:200]}{'...' if len(output_text) > 200 else ''}\n\n"
    summary_md += "---\n\n"

json_md = "# Full Response JSON (with metadata)\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    json_md += f"## Prompt {i}\n\n"
    json_md += f"**Prompt:** {prompt}\n\n"
    json_md += "```json\n"
    json_md += json_lib.dumps(response.model_dump(), indent=2, default=str)
    json_md += "\n```\n\n"

# Update notebook cells directly
import nbformat
from nbformat import v4 as nbf

nb_path = "GPT-OSS-20b_with_vLLM_0.11.2+rhai5.ipynb"
with open(nb_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# Update summary cell (cell index 1)
if len(nb.cells) > 1:
    nb.cells[1].source = summary_md

# Update JSON cell (cell index 2)  
if len(nb.cells) > 2:
    nb.cells[2].source = json_md

with open(nb_path, "w") as f:
    nbformat.write(nb, f)

print("✅ Summary and JSON data added to notebook")


# Run Output Summary

## Prompt 1

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** The uploaded documents contain information about a person named Silas Vance, an Arcturus Systems Engineer who works in the Nexus and NCP team. It describes his current work on manually testing modular...

---

## Prompt 2

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** - Silas Vance is an Arcturus Systems Engineer working with the Nexus and NCP team, focusing on manually testing modular‑stack tools that run on open‑source models using a high‑performance inference en...

---

## Prompt 3

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** The uploaded documents discuss Arcturus Systems Engineering, the Nexus and NCP team, manual testing of modular stack tools with open‑source models on a high‑performance inference engine, and the devel...

---



# Full Response JSON (with metadata)

## Prompt 1

**Prompt:** Use the file search tool to find information in the uploaded documents. What information is available?

```json
{
  "id": "resp_d1972aa2-a046-439c-8912-dccc62f8962a",
  "created_at": 1770141293,
  "model": "vllm/openai/gpt-oss-20b",
  "output": [
    {
      "id": "fc_712e9809-7c9d-4e64-8d16-ef8e551176c4",
      "queries": [
        "information"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "29e40bfc-8585-90d8-00d4-e460cc14bb8a",
            "document_id": "file-840b921e5ade439f89cd55512b7f2a91",
            "token_count": 106.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
          "filename": "file-840b921e5ade439f89cd55512b7f2a91",
          "score": 0.5426211332637317,
          "text": "Silas Vance is an Arcturus Systems Engineer who works in the Nexus and NCP team.\nHe is currently working on manually testing modular stack tools with open-source models on a high-performance inference engine.\nHis manager is Alaric and teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.\nMerrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular-stack responses API."
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded documents contain information about a person named Silas Vance, an Arcturus Systems Engineer who works in the Nexus and NCP team. It describes his current work on manually testing modular stack tools with open\u2011source models on a high\u2011performance inference engine. The manager of Silas Vance is Alaric, and his teammates include Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina. The document notes that Silas Vance and Merrick are planning to build a Test Strategy and Test Automation framework for the modular\u2011stack responses API.",
          "annotations": [
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 141,
              "type": "file_citation"
            },
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 275,
              "type": "file_citation"
            },
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 420,
              "type": "file_citation"
            },
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 573,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_7299e80b-e0f6-4184-a806-3b517db07d01",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 20,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_3041e1eb-d442-42c9-9fa4-b7dedfd0c93c"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 700,
    "output_tokens": 884,
    "total_tokens": 1584,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141309,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 2

**Prompt:** Search the uploaded documents and summarize the key points.

```json
{
  "id": "resp_ac184f61-81c9-4e15-a838-8eeb51111856",
  "created_at": 1770141319,
  "model": "vllm/openai/gpt-oss-20b",
  "output": [
    {
      "id": "fc_31bef446-a1c4-4991-bc7f-269744150405",
      "queries": [
        "uploaded documents key points"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "29e40bfc-8585-90d8-00d4-e460cc14bb8a",
            "document_id": "file-840b921e5ade439f89cd55512b7f2a91",
            "token_count": 106.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
          "filename": "file-840b921e5ade439f89cd55512b7f2a91",
          "score": 0.49739939687875445,
          "text": "Silas Vance is an Arcturus Systems Engineer who works in the Nexus and NCP team.\nHe is currently working on manually testing modular stack tools with open-source models on a high-performance inference engine.\nHis manager is Alaric and teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.\nMerrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular-stack responses API."
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "- Silas\u202fVance is an Arcturus Systems Engineer working with the Nexus and NCP team, focusing on manually testing modular\u2011stack tools that run on open\u2011source models using a high\u2011performance inference engine\u202f  \n- His manager is Alaric and his teammates include Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina\u202f  \n- Silas and Merrick are collaborating to develop a Test Strategy and Test\u2011Automation framework for the modular\u2011stack responses API\u202f.",
          "annotations": [
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 205,
              "type": "file_citation"
            },
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 338,
              "type": "file_citation"
            },
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 472,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_718ba19c-2ec2-4d31-9fd8-24184d37e145",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 20,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_3041e1eb-d442-42c9-9fa4-b7dedfd0c93c"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 690,
    "output_tokens": 520,
    "total_tokens": 1210,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141328,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 3

**Prompt:** What are the main topics covered in the uploaded documents? Use file search to find this information.

```json
{
  "id": "resp_0a0fe738-d693-4265-8d70-1467cc2d29b1",
  "created_at": 1770141338,
  "model": "vllm/openai/gpt-oss-20b",
  "output": [
    {
      "id": "fc_bb8a0581-6c61-4a7a-9ecd-22ff984eebf0",
      "queries": [
        "main topics of uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "29e40bfc-8585-90d8-00d4-e460cc14bb8a",
            "document_id": "file-840b921e5ade439f89cd55512b7f2a91",
            "token_count": 106.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
          "filename": "file-840b921e5ade439f89cd55512b7f2a91",
          "score": 0.5356186783794431,
          "text": "Silas Vance is an Arcturus Systems Engineer who works in the Nexus and NCP team.\nHe is currently working on manually testing modular stack tools with open-source models on a high-performance inference engine.\nHis manager is Alaric and teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.\nMerrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular-stack responses API."
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded documents discuss Arcturus Systems Engineering, the Nexus and NCP team, manual testing of modular stack tools with open\u2011source models on a high\u2011performance inference engine, and the development of a test strategy and test\u2011automation framework for the modular\u2011stack responses API.",
          "annotations": [
            {
              "file_id": "file-840b921e5ade439f89cd55512b7f2a91",
              "filename": "reference.txt",
              "index": 291,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_b8953932-c0f5-47b8-8876-a15f1832254a",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 20,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_3041e1eb-d442-42c9-9fa4-b7dedfd0c93c"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 710,
    "output_tokens": 645,
    "total_tokens": 1355,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141350,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

